# R28 FREE-Replay - Recall Cluster (H293, H294, H298, H299, H300, H311)

**Author**: KGF R28 free-replay executor (recall cluster)
**Date**: 2026-07-09
**Purpose**: Offline, GPU-free replay of six pre-registered R28 hypotheses over the 10 COMPLETED frozen event logs plus frozen reports. No live Neo4j, no GPU. Every verdict is grounded in a cell's own computed output.

**Naive baseline** (reference for every arm): the shipped passive `DriftDetector` (`DriftSettings(remap_threshold=0.3, rebuild_jsd=0.15, window=3)`). Measured pathology across the logs: 52 `drift.warning`, 0 recure, 0 rebuild, even at `js_divergence` 0.5079 (3.4x the 0.15 rebuild threshold), because its window is session-local and resets each run.

Cluster focus: recall/quality decoupling and health-panel structure.

| Arm | Persona | Claim |
|-----|---------|-------|
| H293 | contrarian | active recall@16 probe catches a collapse passive drift misses |
| H294 (br-A) | follower | terminal JSD/remap decoupled from retrieval quality (kill-gate) |
| H298 | mechanist | health panel near rank-deficient; JSD is vehicle not cause |
| H299 | scout | rel-type entropy monitor sees an axis the entity baseline is blind to |
| H300 | scout | resolver-posterior calibration drift is an uncaught health axis |
| H311 | follower | targeted micro-pass beats the full-rebuild floor >=3x recall/token |


## Imports and configuration

In [1]:
import json, glob, math
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr

ROOT = Path('/home/lab/workspace/learning/projects/knowledge-graph-foundry')
LOGS = ['h157','h158-v1','h158-v2','h212-rerun','h212-v2',
        'h240b-enum','h240b-mention','h241-v1','h241-v2','kgf']

def load_events(tag):
    out = []
    with open(ROOT/f'logs/{tag}-events.jsonl') as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: out.append(json.loads(line))
            except json.JSONDecodeError: pass
    return out

EVENTS = {t: load_events(t) for t in LOGS}
def evs(tag, name): return [e for e in EVENTS[tag] if e.get('event')==name]

counts = {t: {'drift.warning': len(evs(t,'drift.warning')),
              'curing.metrics': len(evs(t,'curing.metrics')),
              'drift.decision': len(evs(t,'drift.decision')),
              'resolution.merge': len(evs(t,'resolution.merge'))} for t in LOGS}
print('total curing.metrics rows:', sum(counts[t]['curing.metrics'] for t in LOGS))
pd.DataFrame(counts).T


total curing.metrics rows: 123


,drift.warning,curing.metrics,drift.decision,resolution.merge
h157,0,14,0,2050
h158-v1,0,8,0,2376
h158-v2,0,9,0,732
h212-rerun,0,8,0,347
h212-v2,0,18,0,874
h240b-enum,0,11,0,2217
h240b-mention,0,5,0,2601
h241-v1,3,10,0,525
h241-v2,3,8,0,1016
kgf,46,32,0,4945


## H293 - active recall probe catches what passive drift misses

**Experiment (FREE replay)**: pair `h240b-{enum,mention}-stats.json` + `h241-v{1,2}-stats.json` with matching event logs. Count `drift.warning` per log, simulate a recall@16 trigger at a -0.05 gold-battery threshold, tabulate passive-miss vs active-hit and active false alarms on the flat h241 arms.

**Acceptance bar**: REFUTED unless >=1 recall-drop event (recall drop >=0.05 OR fully_covered drop >=2 on the same corpus) has 0 passive alarms AND the active probe flags it, with 0 active false alarms on the flat h241 arms.

In [2]:
STATS = {t: json.load(open(ROOT/f'reports/{t}-stats.json'))
         for t in ['h240b-enum','h240b-mention','h241-v1','h241-v2']}

enum, mention = STATS['h240b-enum'], STATS['h240b-mention']
rec_drop = enum['mean_recall'] - mention['mean_recall']
cov_drop = enum['fully_covered'] - mention['fully_covered']

v1, v2 = STATS['h241-v1'], STATS['h241-v2']
flat_rec_delta = abs(v1['mean_recall'] - v2['mean_recall'])
flat_cov_delta = abs(v1['fully_covered'] - v2['fully_covered'])

TRIG = 0.05
def passive_alarms(tag): return counts[tag]['drift.warning']
def active_fire(rec_delta, cov_delta): return (rec_delta >= TRIG) or (cov_delta >= 2)

rows = []
rows.append(dict(event='h240b enum->mention', corpus='h240b',
                 recall_drop=round(rec_drop,4), cov_drop=cov_drop,
                 passive_warn_ref=passive_alarms('h240b-enum'),
                 passive_warn_cand=passive_alarms('h240b-mention'),
                 active_fires=active_fire(rec_drop, cov_drop)))
rows.append(dict(event='h241 v1<->v2 (flat)', corpus='h241',
                 recall_drop=round(v2['mean_recall']-v1['mean_recall'],4),
                 cov_drop=v2['fully_covered']-v1['fully_covered'],
                 passive_warn_ref=passive_alarms('h241-v1'),
                 passive_warn_cand=passive_alarms('h241-v2'),
                 active_fires=active_fire(flat_rec_delta, flat_cov_delta)))
h293 = pd.DataFrame(rows)
h293


,event,corpus,recall_drop,cov_drop,passive_warn_ref,passive_warn_cand,active_fires
0,h240b enum->mention,h240b,0.0625,3,0,0,True
1,h241 v1<->v2 (flat),h241,0.0000,0,3,3,False


In [3]:
degr_has_zero_passive = (passive_alarms('h240b-enum')==0 and passive_alarms('h240b-mention')==0)
degr_active_hits = active_fire(rec_drop, cov_drop)
flat_active_false_alarms = 1 if active_fire(flat_rec_delta, flat_cov_delta) else 0
flat_passive_benign = counts['h241-v1']['drift.warning'] + counts['h241-v2']['drift.warning']

confirmed = degr_has_zero_passive and degr_active_hits and flat_active_false_alarms==0
H293 = dict(
    verdict = 'CONFIRMED' if confirmed else 'REFUTED',
    recall_drop = round(rec_drop,4), cov_drop = int(cov_drop),
    passive_alarms_on_degradation = 0,
    active_fires_on_degradation = bool(degr_active_hits),
    active_false_alarms_flat = int(flat_active_false_alarms),
    passive_benign_warnings_flat = int(flat_passive_benign),
)
print(H293)


{'verdict': 'CONFIRMED', 'recall_drop': 0.0625, 'cov_drop': 3, 'passive_alarms_on_degradation': 0, 'active_fires_on_degradation': True, 'active_false_alarms_flat': 0, 'passive_benign_warnings_flat': 6}


## H294 (br-A) - probe battery vs schema-only detector: decoupling kill-gate

**Experiment (br-A FREE kill-gate)**: pair the 6 frozen builds' terminal `js_divergence` (schema-only passive channel) against report `mean_recall`. Spearman `|r| < 0.4` confirms decoupling (arm proceeds); `|r| >= 0.6` kills it. br-B (seeded in-memory render mutation) needs a live render/graph reconstruction and is recorded as queued behind Phase-3.

Terminal remap is only emitted inside `drift.warning`; 4 of the 6 builds fired zero warnings, so remap is structurally unavailable as a terminal channel for those builds - terminal JSD is the available schema-only channel and the named kill-gate.

In [4]:
BUILDS_294 = {
    'h158-v1':'h158-v1-recall.json',
    'h158-v2':'h158-v2-recall.json',
    'h212-v2':'h212-v2-recall.json',
    'h212-rerun':'h212-v2-recall-rerun.json',
    'h241-v1':'h241-v1-recall.json',
    'h241-v2':'h241-v2-recall.json',
}
def terminal_jsd(tag):
    xs = [e['js_divergence'] for e in evs(tag,'curing.metrics')
          if e.get('js_divergence') is not None and not (isinstance(e['js_divergence'],float) and math.isnan(e['js_divergence']))]
    return xs[-1] if xs else np.nan

rows=[]
for tag,rf in BUILDS_294.items():
    r = json.load(open(ROOT/f'reports/{rf}'))
    rows.append(dict(build=tag, terminal_jsd=terminal_jsd(tag),
                     mean_recall=r['mean_recall'], fully_covered=r['fully_covered']))
h294 = pd.DataFrame(rows)
h294


,build,terminal_jsd,mean_recall,fully_covered
0,h158-v1,0.000432,0.8750,20
1,h158-v2,0.002147,0.8333,19
2,h212-v2,0.000548,0.7500,16
3,h212-rerun,0.000548,0.7917,17
4,h241-v1,0.000511,0.7708,17
5,h241-v2,0.000453,0.7708,17


In [5]:
rho_jsd, p_jsd = spearmanr(h294['terminal_jsd'], h294['mean_recall'])
rho_cov, p_cov = spearmanr(h294['terminal_jsd'], h294['fully_covered'])
absr = abs(rho_jsd)
if absr >= 0.6:      v = 'KILLED'
elif absr < 0.4:     v = 'CONFIRMED'
else:                v = 'PARTIAL'
H294 = dict(verdict=v, spearman_jsd_recall=round(rho_jsd,4), abs_r=round(absr,4),
            spearman_jsd_fullcovered=round(rho_cov,4), n_builds=len(h294),
            note='br-B seeded render-mutation arm queues behind Phase-3 (needs in-memory render reconstruction)')
print(H294)


{'verdict': 'CONFIRMED', 'spearman_jsd_recall': np.float64(-0.1471), 'abs_r': np.float64(0.1471), 'spearman_jsd_fullcovered': np.float64(-0.2772), 'n_builds': 6, 'note': 'br-B seeded render-mutation arm queues behind Phase-3 (needs in-memory render reconstruction)'}


## H298 - the health panel is near rank-deficient; JSD is vehicle not cause

**Experiment (FREE replay)**: br1 = all `curing.metrics` rows (18-metric panel) across the 10 logs; br2 = paired report `mean_recall`. Build the matrix -> PCA effective rank; per-run terminal univariate Spearman vs `mean_recall`; partial correlation of terminal JSD with recall given the coverage channel.

**Acceptance bar**: REFUTED unless top-2 PC variance >=0.80 AND coverage recall-R^2 >= 0.9x best-single-metric R^2 AND JSD partial-correlation-given-coverage < 0.15.

In [6]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

METRICS = ['unique_types','total_occurrences','singletons','doubletons',
           'entropy_shannon','entropy_shannon_delta','kl_divergence','js_divergence',
           'type_accumulation_rate','gini_coefficient','zipf_r_squared','heaps_beta',
           'chao1_estimate','chao1_coverage','ace_estimate',
           'entropy_shannon_var','js_divergence_var','gini_coefficient_var']

rows=[]
for t in LOGS:
    for e in evs(t,'curing.metrics'):
        rows.append({'run':t, **{m:e.get(m,np.nan) for m in METRICS}})
M = pd.DataFrame(rows)
print('panel matrix shape:', M.shape[0], 'rows x', len(METRICS), 'metrics')
X = M[METRICS].astype(float)
X = X.fillna(X.mean())
Xs = StandardScaler().fit_transform(X.values)
pca = PCA().fit(Xs)
evr = pca.explained_variance_ratio_
top2 = float(evr[:2].sum())
pr = (evr.sum()**2)/(np.sum(evr**2))
print(f'top-2 PC variance = {top2:.4f}')
print('per-PC EVR (first 6):', np.round(evr[:6],4))
print(f'participation-ratio effective rank = {pr:.2f} of {len(METRICS)}')


panel matrix shape: 123 rows x 18 metrics
top-2 PC variance = 0.5228
per-PC EVR (first 6): [0.3539 0.1689 0.106  0.0837 0.0661 0.0462]
participation-ratio effective rank = 5.45 of 18


In [7]:
RECALL = {
    'h158-v1':0.875,'h158-v2':0.8333,'h212-v2':0.75,'h212-rerun':0.7917,
    'h241-v1':0.7708,'h241-v2':0.7708,'h240b-enum':0.7292,'h240b-mention':0.6667,
}
def terminal_row(t):
    e = evs(t,'curing.metrics')
    return e[-1] if e else None

trows=[]
for t,rec in RECALL.items():
    e = terminal_row(t)
    d = {'run':t,'mean_recall':rec}
    for m in METRICS: d[m]=e.get(m,np.nan)
    trows.append(d)
T = pd.DataFrame(trows)
uni=[]
for m in METRICS:
    col = T[m].astype(float)
    if col.notna().sum() < 4 or col.nunique() < 3:
        uni.append((m, np.nan, np.nan)); continue
    rho,_ = spearmanr(col, T['mean_recall'])
    uni.append((m, rho, rho**2))
U = pd.DataFrame(uni, columns=['metric','spearman','R2']).sort_values('R2', ascending=False)
U


,metric,spearman,R2
9,gini_coefficient,-0.578313,0.334446
1,total_occurrences,-0.554217,0.307156
11,heaps_beta,0.445783,0.198723
10,zipf_r_squared,-0.337349,0.113805
0,unique_types,-0.253012,0.064015
14,ace_estimate,-0.253012,0.064015
12,chao1_estimate,-0.253012,0.064015
13,chao1_coverage,-0.196352,0.038554
2,singletons,-0.194038,0.037651
17,gini_coefficient_var,0.144578,0.020903


In [8]:
cov_channel = 'chao1_coverage'
cov_R2 = float(U.set_index('metric').loc[cov_channel,'R2'])
best_metric = U.iloc[0]['metric']; best_R2 = float(U.iloc[0]['R2'])
cov_spear = float(U.set_index('metric').loc[cov_channel,'spearman'])
jsd_spear = float(U.set_index('metric').loc['js_divergence','spearman'])

def partial_corr(a,b,c,df):
    from numpy.linalg import lstsq
    A=df[a].astype(float).values; B=df[b].astype(float).values; C=df[c].astype(float).values
    C1=np.c_[np.ones_like(C),C]
    ra=A-C1@lstsq(C1,A,rcond=None)[0]
    rb=B-C1@lstsq(C1,B,rcond=None)[0]
    return pearsonr(ra,rb)[0]
jsd_partial = partial_corr('js_divergence','mean_recall',cov_channel,T)

bar_pca   = top2 >= 0.80
bar_cov   = cov_R2 >= 0.9*best_R2
bar_jsd   = abs(jsd_partial) < 0.15
confirmed = bar_pca and bar_cov and bar_jsd
H298 = dict(
    verdict='CONFIRMED' if confirmed else ('PARTIAL' if (bar_pca or bar_jsd) else 'REFUTED'),
    top2_pc_variance=round(top2,4), effective_rank=round(float(pr),2),
    coverage_channel=cov_channel, coverage_R2=round(cov_R2,4), coverage_spearman=round(cov_spear,4),
    best_single_metric=best_metric, best_R2=round(best_R2,4),
    coverage_ge_0p9x_best=bool(bar_cov),
    jsd_terminal_spearman=round(jsd_spear,4),
    jsd_partial_given_coverage=round(float(jsd_partial),4), jsd_partial_lt_0p15=bool(bar_jsd),
    bar_pca=bool(bar_pca))
print(H298)


{'verdict': 'REFUTED', 'top2_pc_variance': 0.5228, 'effective_rank': 5.45, 'coverage_channel': 'chao1_coverage', 'coverage_R2': 0.0386, 'coverage_spearman': -0.1964, 'best_single_metric': 'gini_coefficient', 'best_R2': 0.3344, 'coverage_ge_0p9x_best': False, 'jsd_terminal_spearman': 0.0, 'jsd_partial_given_coverage': 0.366, 'jsd_partial_lt_0p15': False, 'bar_pca': False}


## H299 - relationship-type entropy monitor sees an axis the baseline is blind to

**Experiment (FREE replay)**: recompute a windowed rel-type-entropy CUSUM over the `scorecard-*.json` `relationship_type_entropy` series and set-diff its firings against emitted `drift.decision`. Per-doc rel-TYPE strings are absent from event logs, so the build-level snapshot trend is the free kill-gate (per-doc emit + rebuild arm queues behind Phase-3).

**Acceptance bar**: REFUTED unless the monitor flags >=1 real proliferation build (delta rel-type-entropy > 1.5 nat) the entity-type baseline scored none, with 0 false alarms across stable-entropy builds.

In [9]:
sc_files = sorted(glob.glob(str(ROOT/'reports/scorecard-*.json')))
series=[]
for s in sc_files:
    d=json.load(open(s))
    series.append((Path(s).name, d['scorecard']['relationship_type_entropy']))
sc = pd.DataFrame(series, columns=['scorecard','rel_type_entropy'])
sc


,scorecard,rel_type_entropy
0,scorecard-20260706-090513.json,2.223595
1,scorecard-20260706-105015.json,6.218169
2,scorecard-20260706-120945.json,3.058645
3,scorecard-20260707-091227.json,0.000000
4,scorecard-20260707-115042.json,2.843591


In [10]:
x = sc['rel_type_entropy'].values
baseline_decisions = sum(counts[t]['drift.decision'] for t in LOGS)

ref = np.median(x); k = 0.5; h = 1.5
S=0.0; fires=[]
for i,v in enumerate(x):
    S = max(0.0, S + (v - ref - k))
    if S > h:
        fires.append(i); S = 0.0
deltas = np.diff(x, prepend=x[0])
proliferation_builds = [i for i in fires if deltas[i] > 1.5]
false_alarms = [i for i in fires if not (deltas[i] > 1.5)]

sc2 = sc.copy(); sc2['upward_delta']=np.round(deltas,3); sc2['cusum_fires']=[i in fires for i in range(len(x))]
print(sc2.to_string(index=False))
print('ref(median)=', round(ref,3), ' fires at idx', fires, ' proliferation(>1.5nat)', proliferation_builds,
      ' false_alarms', false_alarms, ' baseline drift.decision total', baseline_decisions)


                     scorecard  rel_type_entropy  upward_delta  cusum_fires
scorecard-20260706-090513.json          2.223595         0.000        False
scorecard-20260706-105015.json          6.218169         3.995         True
scorecard-20260706-120945.json          3.058645        -3.160        False
scorecard-20260707-091227.json          0.000000        -3.059        False
scorecard-20260707-115042.json          2.843591         2.844        False
ref(median)= 2.844  fires at idx [1]  proliferation(>1.5nat) [1]  false_alarms []  baseline drift.decision total 0


In [11]:
confirmed = (len(proliferation_builds) >= 1) and (baseline_decisions == 0) and (len(false_alarms) == 0)
H299 = dict(
    verdict='CONFIRMED' if confirmed else 'REFUTED',
    series=[round(float(v),3) for v in x],
    fires_idx=[int(i) for i in fires],
    proliferation_build_idx=[int(i) for i in proliferation_builds],
    proliferation_delta_nat=round(float(deltas[proliferation_builds[0]]),3) if proliferation_builds else None,
    false_alarms=len(false_alarms),
    baseline_drift_decisions=int(baseline_decisions))
print(H299)


{'verdict': 'CONFIRMED', 'series': [2.224, 6.218, 3.059, 0.0, 2.844], 'fires_idx': [1], 'proliferation_build_idx': [1], 'proliferation_delta_nat': 3.995, 'false_alarms': 0, 'baseline_drift_decisions': 0}


## H300 - temporal calibration (ECE) drift on the resolver posterior stream

**Experiment (FREE replay)**: over `logs/kgf + h241-* + h240b-*` `resolution.merge` posterior fields, compute a per-epoch reliability proxy (binned mean posterior vs merge-acceptance rate per bin) and cross-reference `drift.decision`. True-label ECE against the H157 identity set queues behind Phase-3.

**Acceptance bar**: REFUTED unless the posterior-distribution ECE proxy drifts >=0.05 between logged epochs with 0 corresponding drift alarms.

Note: `resolution.merge` logs accepted merges only (every event `decision == "merge"`), so per-bin acceptance rate = 1.0; the self-consistency ECE proxy reduces to `mean_bin(1 - posterior)` - the gap between predicted merge-confidence and the realized accept outcome.

In [12]:
EPOCHS = ['kgf','h240b-enum','h240b-mention','h241-v1','h241-v2']
def posts(t): return [e['posterior'] for e in evs(t,'resolution.merge')]

def ece_proxy(ps, nbins=10):
    ps=np.array(ps); edges=np.linspace(0,1,nbins+1); tot=len(ps); e=0.0
    for i in range(nbins):
        lo,hi=edges[i],edges[i+1]
        m=(ps>=lo)&(ps<hi if i<nbins-1 else ps<=hi)
        if m.sum()==0: continue
        conf=ps[m].mean(); acc=1.0
        e += (m.sum()/tot)*abs(acc-conf)
    return e

rows=[]
for t in EPOCHS:
    ps=posts(t)
    rows.append(dict(epoch=t, n_merges=len(ps), post_mean=round(float(np.mean(ps)),4),
                     ece_proxy=round(ece_proxy(ps),4), drift_decisions=counts[t]['drift.decision']))
h300 = pd.DataFrame(rows)
h300


,epoch,n_merges,post_mean,ece_proxy,drift_decisions
0,kgf,4945,0.7964,0.2036,0
1,h240b-enum,2217,0.5450,0.4550,0
2,h240b-mention,2601,0.5352,0.4648,0
3,h241-v1,525,0.7226,0.2774,0
4,h241-v2,1016,0.4987,0.5013,0


In [13]:
ece_drift = float(h300['ece_proxy'].max() - h300['ece_proxy'].min())
post_shift = float(h300['post_mean'].max() - h300['post_mean'].min())
total_drift_alarms = int(h300['drift_decisions'].sum())
confirmed = (ece_drift >= 0.05) and (total_drift_alarms == 0)
H300 = dict(
    verdict='CONFIRMED' if confirmed else 'REFUTED',
    ece_proxy_drift=round(ece_drift,4),
    posterior_mean_shift=round(post_shift,4),
    kgf_post_mean=round(float(h300.set_index('epoch').loc['kgf','post_mean']),4),
    h240b_enum_post_mean=round(float(h300.set_index('epoch').loc['h240b-enum','post_mean']),4),
    drift_alarms_total=total_drift_alarms)
print(H300)


{'verdict': 'CONFIRMED', 'ece_proxy_drift': 0.2977, 'posterior_mean_shift': 0.2977, 'kgf_post_mean': 0.7964, 'h240b_enum_post_mean': 0.545, 'drift_alarms_total': 0}


## H311 - targeted-intervention menu beats the full-rebuild floor >=3x per token

**Experiment (FREE simulation)**: full-rebuild recall gain and token cost from census anchors in `unranked-residue-h211` (`census_tokens` 2,140,060; knee 119,843); targeted-menu gain from H271 adjudication (`dw_uniform_ratio_by_budget`: 39 demand-weighted flips at K=1) in `usage-coupling-gates-r26`; price the micro-pass at the knee residue scan and at the H252/H272 <=0.2x re-ingest constant. Compute recall-per-token for both and the ratio.

**Acceptance bar**: REFUTED unless the targeted menu reaches >=90% of the full-rebuild recall gain at <=20% of rebuild tokens (>=3x recall-per-token); also refuted if the missed-probe mass does NOT localize to a repairable neighborhood.

In [14]:
resid = json.load(open(ROOT/'reports/unranked-residue-h211-20260707T190724Z.json'))
usage = json.load(open(ROOT/'reports/usage-coupling-gates-r26-20260708T075940Z.json'))

census_tokens = resid['pinned_baseline']['census_tokens']
knee_tokens   = resid['pinned_baseline']['knee_budget_tokens']
knee_pct      = resid['pinned_baseline']['knee_pct']

flips_by_budget = usage['adjudications']['H271']['dw_uniform_ratio_by_budget']
targeted_flips  = flips_by_budget['1']['demand_weighted']
full_flips      = max(flips_by_budget[k]['demand_weighted'] for k in flips_by_budget)

full_gain, full_cost = full_flips, census_tokens
tgt_gain, tgt_cost_knee = targeted_flips, knee_tokens
tgt_cost_02x = 0.2 * census_tokens

rpt_full = full_gain/full_cost
rpt_tgt_knee = tgt_gain/tgt_cost_knee
rpt_tgt_02x  = tgt_gain/tgt_cost_02x
print(f'full rebuild: gain={full_gain} flips, cost={full_cost:,} tok, recall/tok={rpt_full:.3e}')
print(f'targeted(knee {knee_pct}% ={knee_tokens:,} tok): gain={tgt_gain}, recall/tok={rpt_tgt_knee:.3e}, ratio={rpt_tgt_knee/rpt_full:.2f}x')
print(f'targeted(<=0.2x re-ingest ={tgt_cost_02x:,.0f} tok): gain={tgt_gain}, recall/tok={rpt_tgt_02x:.3e}, ratio={rpt_tgt_02x/rpt_full:.2f}x')


full rebuild: gain=41 flips, cost=2,140,060 tok, recall/tok=1.916e-05
targeted(knee 5.6% =119,843 tok): gain=39, recall/tok=3.254e-04, ratio=16.99x
targeted(<=0.2x re-ingest =428,012 tok): gain=39, recall/tok=9.112e-05, ratio=4.76x


In [15]:
gain_frac = tgt_gain/full_gain
tok_frac_knee = knee_tokens/census_tokens
ratio_knee = rpt_tgt_knee/rpt_full
ratio_02x  = rpt_tgt_02x/rpt_full
localizes = knee_pct <= 20.0
bar = (gain_frac >= 0.90) and (tok_frac_knee <= 0.20) and (ratio_knee >= 3.0) and localizes
H311 = dict(
    verdict='CONFIRMED' if bar else 'REFUTED',
    targeted_flips=int(tgt_gain), full_rebuild_flips=int(full_gain),
    gain_fraction=round(gain_frac,4),
    targeted_token_fraction_knee=round(tok_frac_knee,4),
    recall_per_token_ratio_knee=round(ratio_knee,2),
    recall_per_token_ratio_02x=round(ratio_02x,2),
    census_tokens=census_tokens, knee_tokens=knee_tokens,
    localizes_to_repairable_neighborhood=bool(localizes))
print(H311)


{'verdict': 'CONFIRMED', 'targeted_flips': 39, 'full_rebuild_flips': 41, 'gain_fraction': 0.9512, 'targeted_token_fraction_knee': 0.056, 'recall_per_token_ratio_knee': 16.99, 'recall_per_token_ratio_02x': 4.76, 'census_tokens': 2140060, 'knee_tokens': 119843, 'localizes_to_repairable_neighborhood': True}


## Cluster summary and checkpoint

In [16]:
RUNTAG = 'run01'
summary = {'H293':H293,'H294':H294,'H298':H298,'H299':H299,'H300':H300,'H311':H311}
for kk,vv in summary.items():
    print(kk, '->', vv['verdict'])

out = {
  'cluster':'recall','runtag':RUNTAG,
  'baseline':'passive DriftDetector (remap 0.3, rebuild_jsd 0.15, window 3): 52 drift.warning, 0 recure, 0 rebuild',
  'arms': summary,
}
outpath = ROOT/f'reports/r28-freeplay-recall-{RUNTAG}.json'
json.dump(out, open(outpath,'w'), indent=2, default=str)
print('checkpointed ->', outpath)


H293 -> CONFIRMED
H294 -> CONFIRMED
H298 -> REFUTED
H299 -> CONFIRMED
H300 -> CONFIRMED
H311 -> CONFIRMED
checkpointed -> /home/lab/workspace/learning/projects/knowledge-graph-foundry/reports/r28-freeplay-recall-run01.json
